In [1]:
import warnings
import os
import json
from random import randrange
from functools import partial
import torch
from datasets import Dataset
from datasets import load_dataset
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          HfArgumentParser,
                          Trainer,
                          TrainingArguments,
                          DataCollatorForLanguageModeling,
                          EarlyStoppingCallback,
                          pipeline,
                          logging,
                          set_seed)

import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel, AutoPeftModelForCausalLM
from trl import SFTTrainer
import pandas as pd
from tqdm import tqdm
from utils import write_and_print

In [2]:
LOAD_TESTSET_PATH = '/mnt/fstore/DataFiles/Saved_Datasets/PartyMajorityClassification/Party/Answer/Llama2_Mark0_trainSet.csv'
preprocessed_test_ds = load_dataset('csv', data_files=LOAD_TESTSET_PATH)['train']

In [3]:
DF_PATH = '/mnt/fstore/DataFiles/PickledFiles/Data_MARK2.pkl'
df = pd.read_pickle(DF_PATH)

In [4]:
preprocessed_test_ds

Dataset({
    features: ['Party', 'conversation_id', 'input_ids', 'attention_mask'],
    num_rows: 125032
})

In [5]:
conv_ids = []
for sample in tqdm(preprocessed_test_ds):
    conv_ids.append(sample['conversation_id'])

100%|██████████| 125032/125032 [00:04<00:00, 30685.65it/s]


In [7]:
len(df.loc[conv_ids])

125032

In [8]:
temp_df = df.loc[conv_ids]

In [10]:
temp_df.to_pickle('train_temp.pkl')